# 查询探测：现代 Hopfield 能否知道自己检索错了？

本次是机制筛选实验，不是某篇论文的复现。记忆、温度、更新公式全部固定，只在推理时改变查询。

沿用 1982 notebook 的 `a → b → c → d → e → f`：**生成记忆 → 存储 → 构造查询与探测 → 检索与投票 → 评估 → 绘图**。实现放在共享 `am_bench/reliability.py`，下面先展示一次完整组合，再运行冻结网格。

我们区分两个问题：**诊断**是发现哪些查询容易错；**纠错**是在不知道答案时选择另一个结果。探测不稳定并不意味着一定错，多数一致也不意味着一定对。

先读 [冻结协议](https://github.com/Heptazero/nn-labs/blob/main/associative-memory/benchmarks/am-bench/RELIABILITY_PROTOCOL.md)。主要角半径为 0.10，0.25 是事先声明的敏感性检查；不能看完结果选最好半径。

## 1. 公共实验组件与中文显示

此 notebook 在 Colab 中自动获取固定版本的共享代码，不需要手动上传 `.py` 文件；不安装依赖。本地使用相同源代码指纹时直接复用仓库。数值计算使用已有 PyTorch、CPU、float64、单线程。

中文字体沿用 1982 的 Noto CJK 下载和注册方式。

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile

# [溯源] 固定的是实验源代码版本；不随远端 main 自动改变公式。
CODE_REV = "fc15a37a9657ff017f9da6aff29576e78a96121d"
CODE_PACKAGE_SHA = "845218c945cabf254910bdf47f603343d32c4d40044bc22822af39e18ba5c6bb"
PACKAGE_REL = Path("associative-memory/benchmarks/am-bench/src")

def package_digest(root):
    digest = hashlib.sha256()
    package_root = root / PACKAGE_REL
    for path in sorted((package_root / "am_bench").rglob("*.py")):
        digest.update(str(path.relative_to(package_root)).encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()

# [输入] 本地 notebook 可能从仓库根目录或 notebook 所在目录启动。
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / PACKAGE_REL / "am_bench").is_dir() and package_digest(path) == CODE_PACKAGE_SHA), None)
if ROOT is None:
    ROOT = Path(tempfile.gettempdir()) / ("nn-labs-reliability-" + CODE_REV)
    if not ROOT.exists():
        ROOT.mkdir()
        subprocess.run(["git", "init", str(ROOT)], check=True, capture_output=True)
        subprocess.run(["git", "-C", str(ROOT), "remote", "add", "origin",
                        "https://github.com/Heptazero/nn-labs.git"], check=True, capture_output=True)
    # [约束] 已修改的缓存不能被 checkout 覆盖。
    dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"], text=True)
    if dirty:
        raise RuntimeError("缓存源码已被修改，请使用新运行时或先检查缓存。")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", CODE_REV], check=True, capture_output=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REV], check=True, capture_output=True)
if package_digest(ROOT) != CODE_PACKAGE_SHA:
    raise RuntimeError("源代码指纹不匹配，停止运行。")
PACKAGE_ROOT = ROOT / PACKAGE_REL
if "am_bench" in sys.modules and Path(sys.modules["am_bench"].__file__).resolve().parent != (PACKAGE_ROOT / "am_bench").resolve():
    raise RuntimeError("内核载入了其他版本 am_bench，请重启运行时。")
sys.path.insert(0, str(PACKAGE_ROOT))
print("实验源代码：", CODE_REV)
print("本地路径：", ROOT)

In [ ]:
# [输入] 实验组件：先清楚每个组件负责什么，再组合调用。
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from urllib.request import urlretrieve
from IPython.display import display
from am_bench.reliability import (
    ProbeConfig, a1_make_memories, b1_store_memories,
    c1_make_queries, c2_make_directions, c3_spherical_probes,
    d1_retrieve, d2_majority_repair,
    e1_selective_risk, e2_seed_metrics, e3_paired_intervals, run_probe_grid,
)
from am_bench.provenance import source_info

# [实验控制] 小矩阵在 CPU 单线程上运行，避免线程调度干扰耗时。
torch.set_num_threads(1)
plt.style.use("seaborn-v0_8-whitegrid")
_FONT_PATH = Path("/content/NotoSansCJKtc-Regular.otf") if Path("/content").exists() else Path(tempfile.gettempdir()) / "NotoSansCJKtc-Regular.otf"
_FONT_URL = ("https://raw.githubusercontent.com/notofonts/noto-cjk/main/"
             "Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf")
if not _FONT_PATH.exists():
    urlretrieve(_FONT_URL, _FONT_PATH)
fm.fontManager.addfont(_FONT_PATH)
plt.rcParams["font.family"] = fm.FontProperties(fname=_FONT_PATH).get_name()
plt.rcParams["axes.unicode_minus"] = False
print("中文字体：", plt.rcParams["font.family"])
print("实际载入：", source_info())

## 2. 先看一条查询怎样走完整条路径

`a1 → b1 → c1 → d1 → c2 → c3 → d1 → d2 → e`

先检索一次得到 A，再选择 A 和初始分数最高的另外三条记忆。每个方向去掉径向分量，保证探测只改变方向；正负各走一次，共八次。全部重新从扰动后的原始查询开始检索。

下面固定种子展示第一条查询，**不会搜索一个恰好纠正成功的例子**。目标 ID 在打印评估结果时才使用，不传给方向选择或投票。

In [ ]:
# 1. 构造输入 ----------------------------------------------------------
memories = a1_make_memories(N=32, P=64, kind="close_pairs", seed=3200000)
model = b1_store_memories(memories)
cues, target_ids, cue_names = c1_make_queries(memories, count=1,
                                               kinds=("gaussian_1.6",), seed=3200001)

# 2. 原始检索 → 方向 → 正负探测 → 固定多数票 ---------------------------
base_ids, base_states, residual = d1_retrieve(model, cues, steps=12)
axes, degenerate = c2_make_directions(model, cues, base_ids,
                                     method="guided", count=4, seed=3200010)
probes = c3_spherical_probes(cues, axes, angle=0.10)
probe_ids, _, _ = d1_retrieve(model, probes.flatten(0, 1), steps=12)
probe_ids = probe_ids.reshape(1, 8)
repaired_ids = d2_majority_repair(base_ids, probe_ids, P=64)

# 3. 事后评估：这里才打开答案 -----------------------------------------
display(pd.DataFrame({"真实目标": target_ids.tolist(), "原预测": base_ids.tolist(),
                      "探测预测": probe_ids.tolist(), "多数票预测": repaired_ids.tolist(),
                      "下一步残差": residual.tolist()}))
print("8 票至少 5 票一致才改答；其余情形保留原预测。")

## 3. 冻结网格：32 个记忆库，2048 条查询

两种维度 × 两种记忆结构 × 八个种子，每个库十六个目标、四种噪声条件。每条查询运行三种方向方法、两个半径。`guided` 与 `shuffled` 都包含基线预测记忆，区别只在另外三个候选如何选择。

固定十二步不等于到达固定点；另外一步残差只供观测。108 步单轨迹对照匹配“原检索 + 八条探测”的主要矩阵乘法次数，不包含生成方向、排序和诊断开销，因此不声称墙钟严格相同。

结果在 `results/` 分批保存，源码、配置或依赖版本变化会创建新目录。已完成的相同批次仅复用，不覆盖。Colab 临时运行时释放前，请保存输出 notebook 或下载结果目录。

In [ ]:
# [实验控制] 所有条件在首次运行前写入协议；不在结果出来后改温度或半径。
CONFIG = ProbeConfig()
display(pd.Series(CONFIG.__dict__, name="冻结配置"))

# [运行] a → b → c → d 的网格编排，每个记忆库一个不可覆盖的批次。
RESULTS_ROOT = Path("/content/nn-labs-results/retrieval-reliability") if Path("/content").exists() else Path(tempfile.gettempdir()) / "nn-labs-results/retrieval-reliability"
rows, RUN_DIR, run_info = run_probe_grid(CONFIG, RESULTS_ROOT)
raw = pd.DataFrame(rows)
expected = (len(CONFIG.dimensions) * len(CONFIG.memory_kinds) * len(CONFIG.seeds)
            * CONFIG.targets * len(CONFIG.cue_kinds) * len(CONFIG.methods) * len(CONFIG.angles))
assert len(raw) == expected == 12288
assert not raw.duplicated(["N", "memory_kind", "seed", "query_index", "method", "angle"]).any()
print("结果目录：", RUN_DIR)
print(run_info)

## 4. 诊断：拒绝最不可信的 20%，留下来的错误更少吗？

`e1 → e2 → e3 → f1`

每个库中按风险从低到高保留 80% 查询。探测量只有九种可能取值，并列会很常见；边界并列按等概率抽取的期望计分，不能利用真实标签打破并列。错误率越低越好。

图中是 **guided 的错误率减去对照错误率**，负值有利。每个格子先按种子配对，再取八个种子的均值；详细区间保存在配对表中。干净条件的零差值可能只是没有错误，不表示诊断有效。

In [ ]:
# [观测·数据] 保存可重建统计；不把同一记忆库中的查询当作独立实验。
seed_metrics = e2_seed_metrics(rows)
paired = e3_paired_intervals(seed_metrics, angle=0.10)
seed_metrics.to_csv(RUN_DIR / "seed_metrics.csv", index=False)
paired.to_csv(RUN_DIR / "paired_intervals.csv", index=False)

# [展示] 条件名保持精确，颜色统一以百分比点为单位。
def f1_diagnostic_comparison(paired):
    table = paired.pivot(index=["N", "memory_kind", "cue_kind"],
                         columns="control", values="risk_difference")
    table = table[["gap", "entropy", "random", "shuffled"]] * 100
    fig, ax = plt.subplots(figsize=(10, 8))
    limit = max(float(np.abs(table.to_numpy()).max()), 0.1)
    ax.grid(False)
    im = ax.imshow(table, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    ax.set_xticks(range(4), ["初始分数间隔", "初始注意力熵", "随机方向", "随机候选"])
    ax.set_yticks(range(len(table)), [f"N={n} / {m} / {c}" for n, m, c in table.index])
    for (i, j), value in np.ndenumerate(table.to_numpy()):
        ax.text(j, i, f"{value:+.2f}", ha="center", va="center", fontsize=9,
                color="white" if abs(value) > 0.6 * limit else "black")
    ax.set_title("保留 80% 查询后的错误率差：guided − 对照\n角半径 0.10；负值有利")
    fig.colorbar(im, ax=ax, label="错误率差（百分点）")
    fig.tight_layout()
    fig.savefig(RUN_DIR / "diagnostic.png", dpi=150)
    return fig

f1_diagnostic_comparison(paired)
plt.show()
display(paired.round(4))

**这张图能支持什么：**比较的是这组查询条件下的诊断排序，不是每个查询的错误概率。只有比便宜的分数间隔更好，探测成本才可能值得。区间只反映八个记忆库的波动，不能代替新数据集上的验证。

## 5. 纠错：救回多少，又误伤多少？

`d2 → e2 → f2`

投票规则在看标签前固定。净收益 = 错改对数量 − 对改错数量，再除以查询数。不同错误记忆之间的改动没有收益。图同时展示两个预定半径，不选择其中较好的一个作为主要结果。

下表加入原始查询最近邻、十二步 Hopfield、108 步 Hopfield。即使投票改善了十二步结果，也要检查它是否超过最简单的最近邻。

In [ ]:
def f2_repair_gain(seed_metrics):
    table = seed_metrics.groupby(["N", "memory_kind", "cue_kind", "method", "angle"])["net_gain"].mean().unstack(["method", "angle"])
    table *= 100
    fig, ax = plt.subplots(figsize=(11, 8))
    limit = max(float(np.abs(table.to_numpy()).max()), 0.1)
    ax.grid(False)
    im = ax.imshow(table, cmap="RdBu", vmin=-limit, vmax=limit, aspect="auto")
    ax.set_xticks(range(len(table.columns)), [f"{m}\n{a:.2f}" for m, a in table.columns])
    ax.set_yticks(range(len(table)), [f"N={n} / {m} / {c}" for n, m, c in table.index])
    for (i, j), value in np.ndenumerate(table.to_numpy()):
        ax.text(j, i, f"{value:+.2f}", ha="center", va="center", fontsize=9,
                color="white" if abs(value) > 0.6 * limit else "black")
    ax.set_title("严格多数票的净正确率变化：正值有利")
    fig.colorbar(im, ax=ax, label="正确率变化（百分点）")
    fig.tight_layout()
    fig.savefig(RUN_DIR / "repair.png", dpi=150)
    return fig

f2_repair_gain(seed_metrics)
plt.show()
primary = seed_metrics[(seed_metrics.method == "guided") & np.isclose(seed_metrics.angle, 0.10)]
comparison = primary.groupby(["N", "memory_kind", "cue_kind"]).agg(
    查询数=("queries", "sum"), 最近邻正确率=("nearest_accuracy", "mean"),
    原检索正确率=("baseline_accuracy", "mean"), 延长检索正确率=("extended_accuracy", "mean"),
    投票正确率=("repaired_accuracy", "mean"), 错改对=("rescued", "sum"), 对改错=("harmed", "sum"),
    基线收敛比例=("baseline_converged", "mean"), 探测收敛比例=("probe_converged", "mean"))
display(comparison.round(4))
comparison.to_csv(RUN_DIR / "primary_comparison.csv")

## 6. 怎样判断该继续还是停止

先看 guided 是否优于分数间隔和随机对照，再看净纠错是否为正，最后检查最近邻是否已经更好。这里没有训练标签，也没有参数更新，但仍有固定的算法选择和超参数；“不训练”不等于“不做任何假设”。

如果只能诊断而不能纠错，最多支持拒答/重试的用途；如果连诊断也没有额外信息，就停止这条探测规则。没有真实数据和经典 Hopfield 的结果，不能推广为所有关联记忆都需要这种机制。

最终结果解读见同目录 `RELIABILITY_RESULTS.md`。本 notebook 的中文绘图与数值运行可在本地验证；Colab 页面上的点击验收由你完成。

**2026-09-06 实际结果：**完整网格 12288 条记录，主要 guided/0.10 在 2048 条查询中错改对 0 条、对改错 1 条；所有方法和半径均没有救回错误。N=128 的块污染条件下，guided 比随机探测更能筛出错误，但独立记忆上不如初始分数间隔。近邻成对记忆上相对间隔的风险差为 −0.98 个百分点，探索性 95% 区间跨零。

N=32 的十二步读出连干净查询也只能达到独立记忆 2.34%、成对记忆 1.56% 的身份正确率；该区域的基线检索已失效。N=128 的块污染条件下，原始最近邻也优于十二步检索。故本轮停止多数票规则，不把它当成有效的自适应检索方法。这里没有实现可学习或自适应的相似函数。
